In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.embeddings import NILCEmbeddings
from src.vectorstore import carregar_colecao

In [2]:
embeddings = NILCEmbeddings()
store = carregar_colecao(embeddings)

print("Registros no Chroma:", store._collection.count())

emb = np.load("../data/processed/embeddings.npy")
df = pd.read_parquet("../data/processed/reviews_indexed.parquet")

pergunta = "atraso na entrega do pedido"

print("=== Chroma ===")
for doc, dist in store.similarity_search_with_score(pergunta, k=5):
    print(f"sim={1 - dist:.3f}  [{doc.metadata['review_score']}★] {doc.page_content}")

print("\n=== Gabarito (NumPy, busca exata) ===")
q = np.array(embeddings.embed_query(pergunta))
sims = emb @ q
for i in np.argsort(-sims)[:5]:
    print(f"sim={sims[i]:.3f}  [{df.loc[i, 'review_score']}★] {df.loc[i, 'texto']}")

Registros no Chroma: 40393
=== Chroma ===
sim=0.932  [1★] Atraso na entrega
sim=0.932  [2★] Atraso na entrega.
sim=0.932  [4★] Atraso na entrega.
sim=0.932  [1★] ATRASO NA ENTREGA
sim=0.932  [1★] Atraso na entrega.

=== Gabarito (NumPy, busca exata) ===
sim=0.932  [1★] Atraso na entrega.
sim=0.932  [4★] Atraso na entrega.
sim=0.932  [1★] ATRASO NA ENTREGA
sim=0.932  [1★] Atraso na entrega.
sim=0.932  [2★] Atraso na entrega


devido ao embedding, todas as frases atraso na entrega vira o mesmo vetor e da empate, por isso no gabarito temos documentos diferentes

mantivemos os textos duplicados, isso pode ser a causa do top 5 ter pouca diversidade

In [4]:
norm = df["texto"].str.lower().str.replace(r"[^\w\s]", "", regex=True).str.strip()
print("Comentários equivalentes a 'atraso na entrega':", (norm == "atraso na entrega").sum())

Comentários equivalentes a 'atraso na entrega': 9


In [5]:
perguntas = [
    "atraso na entrega do pedido",
    "produto chegou com defeito",
    "ótimo atendimento do vendedor",
    "não recebi o produto",
]

for p in perguntas:
    q = np.array(embeddings.embed_query(p))
    ref = np.sort(emb @ q)[::-1][:10]
    chroma = np.array(sorted(
        [1 - d for _, d in store.similarity_search_with_score(p, k=10)], reverse=True
    ))
    print(f"{p!r:40} diferença máxima (top-10): {np.abs(ref - chroma).max():.5f}")

'atraso na entrega do pedido'            diferença máxima (top-10): 0.00000
'produto chegou com defeito'             diferença máxima (top-10): 0.00000
'ótimo atendimento do vendedor'          diferença máxima (top-10): 0.00000
'não recebi o produto'                   diferença máxima (top-10): 0.00000


a diferença mostra que no top 10 o mesmo comentario apareceria varias vezes, a depender da frequencia.